## Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

pd.set_option('display.max_columns', None)

from src.get_tables import tickers
from src.model_functions import build_lstm_sequences, split_temporal, generate_lstm_model, get_study_optuna

## Funções

In [3]:
save_folder_path_refined = 'dados/refined'

features = [
    'close', 'high', 'low', 'open', 'volume',
    'close_dolar', 'close_ibovespa', 'close_sp_500', 'selic', 'ipca',
    'ma20', 'ma50', 'bb_upper', 'bb_lower', 'rsi_wilder', 'macd',
    'macd_signal', 'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos'
]

In [4]:
tickers

['RENT3', 'LREN3', 'NATU3', 'SMFT3', 'MULT3', 'VBBR3', 'ABEV3']

In [ ]:
tabela_hiperparams = pd.DataFrame()

for t in [t for t in tickers if t != 'ABEV3']:

    df = pd.read_parquet(f"{save_folder_path_refined}/tb_analitica_{t}.parquet", engine = 'pyarrow')
    df = df.sort_values("date").reset_index(drop=True)

    data = df[features].copy()

    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(data)

    for w in [60, 90, 120, 180]:
        print("=" * 60)
        print(f"Início dos estudos | Ticker = {t} | window_size = {w}")
        print("=" * 60)

        study = get_study_optuna(w, scaled_data, n_trials = 20, ticker = t)

        tmp = pd.DataFrame(data = {
            'ticker':[t],
            'window_size':[w],
            'val_loss':[study.best_value]

        })

        for k, v in study.best_params.items():
            tmp[k] = [v]

        tabela_hiperparams = pd.concat([tabela_hiperparams, tmp])
        print("\n\n")

[I 2026-07-06 13:17:55,547] A new study created in memory with name: LSTM Stock Prediction | window_size = 60


Início dos estudos | Ticker = RENT3 | window_size = 60
(2409, 60, 21)
(2409,)
Treino     : (1686, 60, 21)
Validação  : (361, 60, 21)
Teste      : (362, 60, 21)


  0%|          | 0/20 [00:00<?, ?it/s]

g:\Meu Drive\5. Cursos\Pós ML Engineering\Fase 4 - Deep Learning e IA\lstm-stock-predictor-api\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
tabela_hiperparams

In [ ]:
tabela_hiperparams.to_excel('dados/aux_data/tabela_hiperparametros_tickers.xlsx', index = False)